<a href="https://colab.research.google.com/github/Mahsa-Goudarzi/vap-time-indexed/blob/main/vap_notebooks/vap_time_indexed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pulp

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# DATA
# ═══════════════════════════════════════════════════════════════════════════

# carriers and trips
carriers = [0, 1]
trips    = [0, 1, 2, 3, 4]

# Each trip belongs to exactly one carrier  →  sum_r z[r,n] = 1 for all n
trips_of_carrier = {
    0: [0, 1],
    1: [2, 3, 4],
}

vehicles_of_carrier = {
    0: [0, 1, 2],   # V^r for r=0
    1: [3, 4, 5],   # V^r for r=1
}
is_electric = {0: False, 1: False, 2: True,
               3: True,  4: True,  5: False}
all_vehicles = [v for vlist in vehicles_of_carrier.values() for v in vlist]

# ── scalar parameters ──────────────────────────────────────────────────────
alpha = 1.0   # revenue scaling coefficient
beta  = 0.8   # emission cost attribution coefficient
mu    = 80.0  # average truck speed  (km/h)
c_E   = 0.2   # cost per gram CO2    (€/g)

# unit revenue I^r  (€/km)
I = {0: 1.8, 1: 1.8}

# trip distances  d_n  (km)
d = {0: 150, 1: 100, 2: 80, 3: 120, 4: 130}

# unit operating cost  c^{rv}_O  (€/km)
# for vehicles NOT owned by carrier r → large penalty (1000) prevents use
op_cost_all_diesels = 0.8
op_cost_all_electrics = 0.7

op_cost = {
    0: {0: op_cost_all_diesels, 1: op_cost_all_diesels, 2: op_cost_all_electrics},   # carrier 0 owns V0,V1,V2
    1: {3: op_cost_all_electrics, 4: op_cost_all_electrics, 5: op_cost_all_diesels},   # carrier 1 owns V3,V4,V5
}
op_cost_full = {
    (0, 0): op_cost_all_diesels, (0, 1): op_cost_all_diesels, (0, 2): op_cost_all_electrics,
    (0, 3): 1000, (0, 4): 1000, (0, 5): 1000,
    (1, 0): 1000, (1, 1): 1000, (1, 2): 1000,
    (1, 3): op_cost_all_electrics,  (1, 4): op_cost_all_electrics,  (1, 5): op_cost_all_diesels,
}

# available time T^v  (hours)
# diesel  → T^v = T^v_drive
# electric → T^v = min(T^v_drive, T^v_charge)
T_drive  = {v: 8.0 for v in all_vehicles}
T_charge = {v: 8.0 for v in all_vehicles if is_electric[v]}
T_avail  = {}
for v in all_vehicles:
    if is_electric[v]:
        T_avail[v] = min(T_drive[v], T_charge[v])
    else:
        T_avail[v] = T_drive[v]

# vehicle capacity  Cap^v  (kg)
Cap = {v: 40.0 for v in all_vehicles}

# weight of goods in trip n  p_n  (kg)
p = {n: 25.0 for n in trips}

# emission parameters  (g/km)
E_empty = {0: 0.3, 1: 0.2, 2: 0.3, 3: 0.3, 4: 0.3, 5: 0.3}
E_full  = {0: 0.3, 1: 0.4, 2: 0.3, 3: 0.3, 4: 0.4, 5: 0.3}

# d^v_{o_n}: depot of vehicle v → origin of trip n   (km)   key = (n, v)
d_origin = {
    (0,0):0,   (0,1):0,   (0,2):0,   (0,3):50,  (0,4):50,  (0,5):50,
    (1,0):0,   (1,1):0,   (1,2):0,   (1,3):50,  (1,4):50,  (1,5):50,
    (2,0):100, (2,1):100, (2,2):100, (2,3):80,  (2,4):80,  (2,5):80,
    (3,0):50,  (3,1):50,  (3,2):50,  (3,3):0,   (3,4):0,   (3,5):0,
    (4,0):50,  (4,1):50,  (4,2):50,  (4,3):0,   (4,4):0,   (4,5):0,
}

# d^v_{w_k}: destination of trip k → depot of vehicle v  (km)  key = (k, v)
d_dest = {
    (0,0):150, (0,1):150, (0,2):150, (0,3):200, (0,4):200, (0,5):200,
    (1,0):100, (1,1):100, (1,2):100, (1,3):150, (1,4):150, (1,5):150,
    (2,0):50,  (2,1):50,  (2,2):50,  (2,3):0,   (2,4):0,   (2,5):0,
    (3,0):170, (3,1):170, (3,2):170, (3,3):120, (3,4):120, (3,5):120,
    (4,0):180, (4,1):180, (4,2):180, (4,3):130, (4,4):130, (4,5):130,
}

# epsilon_{nk}: repositioning distance dest(n) → origin(k)  (km)
eps = {
    (0,0):0,   (0,1):150, (0,2):60,  (0,3):140, (0,4):140,
    (1,0):150, (1,1):0,   (1,2):0,   (1,3):80,  (1,4):80,
    (2,0):50,  (2,1):50,  (2,2):0,   (2,3):0,   (2,4):0,
    (3,0):170, (3,1):200, (3,2):200, (3,3):0,   (3,4):120,
    (4,0):180, (4,1):180, (4,2):140, (4,3):130, (4,4):0,
}

# O_{nk}: 1 if trips n and k may be combined
O = {
    (0,0):0, (0,1):0, (0,2):1, (0,3):0, (0,4):0,
    (1,0):0, (1,1):0, (1,2):1, (1,3):0, (1,4):0,
    (2,0):1, (2,1):1, (2,2):0, (2,3):1, (2,4):1,
    (3,0):0, (3,1):0, (3,2):1, (3,3):0, (3,4):0,
    (4,0):0, (4,1):0, (4,2):1, (4,3):0, (4,4):0,
}

# delta_n: compensation cost for trip n  (€)
delta = {0: 20, 1: 20, 2: 10, 3: 10, 4: 10}

# z^r_n: 1 if trip n belongs to carrier r
z = {(r, n): (1 if n in trips_of_carrier[r] else 0)
     for r in carriers for n in trips}

# ── time index parameters (new) ──────────────────────────────────────────────────
periods = [1, 2, 3]   # T = {1, 2, 3}

# tau_n: planned execution day of trip n
tau = {0: 1, 1: 1, 2: 2, 3: 2, 4: 3}

# pi_n: penalty coefficient  (€ per day of deviation)
pi = {0: 10.0, 1: 8.0, 2: 12.0, 3: 9.0, 4: 11.0}

# Delta_{nt} = |t - tau_n|  — pre-computed parameter (model stays linear)
Delta = {(n, t): abs(t - tau[n]) for n in trips for t in periods}

print("Time parameters:")
for n in trips:
    print(f"  Trip {n}: tau={tau[n]}, pi={pi[n]}, "
          f"Delta={[Delta[(n,t)] for t in periods]}")


Time parameters:
  Trip 0: tau=1, pi=10.0, Delta=[0, 1, 2]
  Trip 1: tau=1, pi=8.0, Delta=[0, 1, 2]
  Trip 2: tau=2, pi=12.0, Delta=[1, 0, 1]
  Trip 3: tau=2, pi=9.0, Delta=[1, 0, 1]
  Trip 4: tau=3, pi=11.0, Delta=[2, 1, 0]


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# AUXILIARY FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════

def TOA_single(v, n):
    """TOA^v_n = d^v_{o_n} + d_n + d^v_{w_n}"""
    return d_origin[(n, v)] + d[n] + d_dest[(n, v)]

def TOA_combined(v, n, k):
    """TOA^v_{nk} = d^v_{o_n} + d_n + eps_{nk} + d_k + d^v_{w_k}
    Note: depot leg at the end uses destination of trip k, not n."""
    return d_origin[(n, v)] + d[n] + eps[(n, k)] + d[k] + d_dest[(k, v)]

def emission_single(v, n):
    """E^v_n: zero for electric, activity-based for diesel."""
    if is_electric[v]:
        return 0.0
    return (TOA_single(v, n) * E_empty[v]
            + (E_full[v] - E_empty[v]) * d[n] * p[n] / Cap[v])

def emission_combined(v, n, k):
    """E^v_{nk}: zero for electric, activity-based for diesel."""
    if is_electric[v]:
        return 0.0
    return (TOA_combined(v, n, k) * E_empty[v]
            + (E_full[v] - E_empty[v]) * (d[n]*p[n] + d[k]*p[k]) / Cap[v])

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PROBLEM 1 - Initial profit per carrier  (with time indexing)
# ═══════════════════════════════════════════════════════════════════════════
# Decision variable: chi^{rv}_{nt} in {0,1}
# Objective:  max  sum_{n,v,t} (alpha*d_n*I^r - TOA*c_O - c_E*E - pi_n*Delta_{nt}) * chi
# P1-C1: sum_{v,t} chi^{rv}_{nt} = 1          for all n in N^r
# P1-C2: sum_n chi^{rv}_{nt} <= 1              for all v, t
# P1-C3: sum_{n,v} chi^{rv}_{nt} <= V^r        for all t
# P1-C4: chi^{rv}_{nt} * TOA^v_n <= T^v * mu   for all n, v in V^r_E, t

S0 = {}

for r in carriers:
    prob1 = pulp.LpProblem(f"P1_C{r}", pulp.LpMaximize)
    my_trips = trips_of_carrier[r]
    my_vehs  = vehicles_of_carrier[r]
    V_r      = len(my_vehs)

    chi = pulp.LpVariable.dicts(
        "chi",
        [(n, v, t) for n in my_trips for v in my_vehs for t in periods],
        cat="Binary"
    )

    # Objective - eq. (p1_obj)
    prob1 += pulp.lpSum(
        (
            alpha * d[n] * I[r]              # revenue
            - TOA_single(v, n) * op_cost[r][v]  # operating cost
            - c_E * emission_single(v, n)        # emission cost
            - pi[n] * Delta[(n, t)]              # time penalty
        ) * chi[(n, v, t)]
        for n in my_trips for v in my_vehs for t in periods
    )

    # P1-C1: each trip executed exactly once on exactly one day
    for n in my_trips:
        prob1 += (
            pulp.lpSum(chi[(n, v, t)] for v in my_vehs for t in periods) == 1
        ), f"P1C1_n{n}"

    # P1-C2: each vehicle executes at most one trip per day
    for v in my_vehs:
        for t in periods:
            prob1 += (
                pulp.lpSum(chi[(n, v, t)] for n in my_trips) <= 1
            ), f"P1C2_v{v}_t{t}"

    # P1-C3: trips assigned on day t <= fleet size V^r
    for t in periods:
        prob1 += (
            pulp.lpSum(chi[(n, v, t)]
                       for n in my_trips for v in my_vehs) <= V_r
        ), f"P1C3_t{t}"

    # P1-C4: battery range for electric vehicles
    for n in my_trips:
        for v in my_vehs:
            if is_electric[v]:
                for t in periods:
                    prob1 += (
                        chi[(n, v, t)] * TOA_single(v, n) <= T_avail[v] * mu
                    ), f"P1C4_n{n}_v{v}_t{t}"

    prob1.solve(pulp.PULP_CBC_CMD(msg=0))
    S0[r] = pulp.value(prob1.objective)

    print(f"\nCarrier {r}: S0 = {S0[r]:.2f} €  [{pulp.LpStatus[prob1.status]}]")
    for n in my_trips:
        for v in my_vehs:
            for t in periods:
                if pulp.value(chi[(n, v, t)]) and pulp.value(chi[(n, v, t)]) > 0.5:
                    vtype = "E" if is_electric[v] else "D"
                    pen   = pi[n] * Delta[(n, t)]
                    print(f"  Trip {n} (tau={tau[n]}) → V{v}({vtype}), "
                          f"Day {t} | penalty = {pen:.1f} €")




Carrier 0: S0 = 92.00 €  [Optimal]
  Trip 0 (tau=1) → V2(E), Day 1 | penalty = 0.0 €
  Trip 1 (tau=1) → V2(E), Day 2 | penalty = 8.0 €

Carrier 1: S0 = 132.00 €  [Optimal]
  Trip 2 (tau=2) → V3(E), Day 2 | penalty = 0.0 €
  Trip 3 (tau=2) → V4(E), Day 2 | penalty = 0.0 €
  Trip 4 (tau=3) → V4(E), Day 3 | penalty = 0.0 €


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PROBLEM 2 - Cooperative optimisation  (with time indexing)
# ═══════════════════════════════════════════════════════════════════════════
# Decision variables:
#   x^{rv}_{nt}   in {0,1}  - single trip n by carrier r, vehicle v, day t
#   y^{rv}_{nkt}  in {0,1}  - combined trip (n,k) by carrier r, vehicle v, day t

prob2 = pulp.LpProblem("P2_Cooperative_Time", pulp.LpMaximize)

x = pulp.LpVariable.dicts(
    "x",
    [(r, v, n, t)
     for r in carriers
     for v in vehicles_of_carrier[r]
     for n in trips
     for t in periods],
    cat="Binary"
)

y = pulp.LpVariable.dicts(
    "y",
    [(r, v, n, k, t)
     for r in carriers
     for v in vehicles_of_carrier[r]
     for n in trips
     for k in trips if k != n
     for t in periods],
    cat="Binary"
)


def S1(r):
    """S^r_1 - profit from single trips including time penalty.
    eq. (S1): sum_{n,v,t} (alpha*d_n*I^r - TOA*c_O - beta*c_E*E - pi_n*Delta_{nt}) * x
    """
    terms = []
    for n in trips:
        for v in vehicles_of_carrier[r]:
            for t in periods:
                rev      = alpha * d[n] * I[r]
                op_c     = TOA_single(v, n) * op_cost_full[(r, v)]
                em_c     = beta * c_E * emission_single(v, n)
                time_pen = pi[n] * Delta[(n, t)]
                terms.append((rev - op_c - em_c - time_pen) * x[(r, v, n, t)])
    return pulp.lpSum(terms)


def S2(r):
    """S^r_2 - profit from combined trips including time penalties for both trips.
    eq. (S2): sum_{n,k!=n,v,t} (alpha*(d_n+d_k)*I^r - TOA*c_O - beta*c_E*E
                                 - pi_n*Delta_{nt} - pi_k*Delta_{kt}) * y
    Note: both trips incur their own penalty based on the shared execution day t.
    """
    terms = []
    for n in trips:
        for k in trips:
            if k == n:
                continue
            for v in vehicles_of_carrier[r]:
                for t in periods:
                    rev      = alpha * (d[n] + d[k]) * I[r]
                    op_c     = TOA_combined(v, n, k) * op_cost_full[(r, v)]
                    em_c     = beta * c_E * emission_combined(v, n, k)
                    time_pen = pi[n] * Delta[(n, t)] + pi[k] * Delta[(k, t)]
                    terms.append(
                        (rev - op_c - em_c - time_pen) * y[(r, v, n, k, t)]
                    )
    return pulp.lpSum(terms)


def S3(r):
    """S^r_3 - compensation mechanism.
    eq. (S3): delta_n is a trip property, independent of the execution day.
    Only the summation index t is added vs. the base formulation.
    """
    terms = []
    # single trips
    for n in trips:
        for rp in carriers:
            if rp == r:
                continue
            # r' executes trip n (owned by r) → r earns delta_n
            for v in vehicles_of_carrier[rp]:
                for t in periods:
                    terms.append(+delta[n] * z[(r, n)]  * x[(rp, v, n, t)])
            # r executes trip n (owned by r') → r pays delta_n
            for v in vehicles_of_carrier[r]:
                for t in periods:
                    terms.append(-delta[n] * z[(rp, n)] * x[(r, v, n, t)])
    # combined trips
    for n in trips:
        for k in trips:
            if k == n:
                continue
            for rp in carriers:
                if rp == r:
                    continue
                for v in vehicles_of_carrier[rp]:
                    for t in periods:
                        terms.append(
                            +delta[n] * z[(r, n)] * y[(rp, v, n, k, t)])
                        terms.append(
                            +delta[k] * z[(r, k)] * y[(rp, v, n, k, t)])
                for v in vehicles_of_carrier[r]:
                    for t in periods:
                        terms.append(
                            -delta[n] * z[(rp, n)] * y[(r, v, n, k, t)])
                        terms.append(
                            -delta[k] * z[(rp, k)] * y[(r, v, n, k, t)])
    return pulp.lpSum(terms)


# ── Objective - eq. (p2_obj) ────────────────────────────────────────────────
prob2 += pulp.lpSum(S1(r) + S2(r) + S3(r) for r in carriers)

# ── Constraints ─────────────────────────────────────────────────────────────

# P2-C1: final profit >= initial profit  (individual rationality)
for r in carriers:
    prob2 += (
        S1(r) + S2(r) + S3(r) >= S0[r]
    ), f"P2C1_r{r}"

# P2-C2: each trip covered exactly once on exactly one day
for n in trips:
    prob2 += (
        pulp.lpSum(
            x[(r, v, n, t)]
            for r in carriers
            for v in vehicles_of_carrier[r]
            for t in periods
        )
        + pulp.lpSum(
            y[(r, v, n, k, t)] + y[(r, v, k, n, t)]
            for r in carriers
            for v in vehicles_of_carrier[r]
            for k in trips if k != n
            for t in periods
        )
        == 1
    ), f"P2C2_n{n}"

# P2-C3: each vehicle executes at most one trip (or pair) per day
for r in carriers:
    for v in vehicles_of_carrier[r]:
        for t in periods:
            prob2 += (
                pulp.lpSum(x[(r, v, n, t)] for n in trips)
                + pulp.lpSum(
                    y[(r, v, n, k, t)]
                    for n in trips for k in trips if k != n
                )
                <= 1
            ), f"P2C3_r{r}_v{v}_t{t}"

# P2-C4: trips assigned to carrier r on day t <= fleet size V^r
# (implied by P2-C3 but stated explicitly for clarity)
for r in carriers:
    V_r = len(vehicles_of_carrier[r])
    for t in periods:
        prob2 += (
            pulp.lpSum(
                x[(r, v, n, t)]
                for v in vehicles_of_carrier[r]
                for n in trips
            )
            + pulp.lpSum(
                y[(r, v, n, k, t)]
                for v in vehicles_of_carrier[r]
                for n in trips for k in trips if k != n
            )
            <= V_r
        ), f"P2C4_r{r}_t{t}"

# P2-C5: battery range - single trip
for r in carriers:
    for v in vehicles_of_carrier[r]:
        if is_electric[v]:
            for n in trips:
                for t in periods:
                    prob2 += (
                        x[(r, v, n, t)] * TOA_single(v, n) <= T_avail[v] * mu
                    ), f"P2C5_r{r}_v{v}_n{n}_t{t}"

# P2-C6: battery range - combined trip
for r in carriers:
    for v in vehicles_of_carrier[r]:
        if is_electric[v]:
            for n in trips:
                for k in trips:
                    if k != n:
                        for t in periods:
                            prob2 += (
                                y[(r, v, n, k, t)] * TOA_combined(v, n, k)
                                <= T_avail[v] * mu
                            ), f"P2C6_r{r}_v{v}_n{n}_k{k}_t{t}"

# P2-C7: combinability
for r in carriers:
    for v in vehicles_of_carrier[r]:
        for n in trips:
            for k in trips:
                if k != n:
                    for t in periods:
                        prob2 += (
                            y[(r, v, n, k, t)] <= O[(n, k)]
                        ), f"P2C7_r{r}_v{v}_n{n}_k{k}_t{t}"

# ── Solve ───────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("Solving Problem 2 (Cooperative + Time-Indexed)...")
prob2.solve(pulp.PULP_CBC_CMD(msg=0))

print(f"Status : {pulp.LpStatus[prob2.status]}")
print(f"Total Profit: {pulp.value(prob2.objective):.2f} €\n")

for r in carriers:
    s1v = pulp.value(S1(r))
    s2v = pulp.value(S2(r))
    s3v = pulp.value(S3(r))
    total = s1v + s2v + s3v
    print(f"Carrier {r}: S1={s1v:.2f}  S2={s2v:.2f}  S3={s3v:.2f}  "
          f"Total={total:.2f}  (min S0={S0[r]:.2f})")

print("\nTrip Assignments:")
total_penalty = 0.0
for r in carriers:
    for v in vehicles_of_carrier[r]:
        vtype = "E" if is_electric[v] else "D"
        # single trips
        for n in trips:
            for t in periods:
                val = pulp.value(x[(r, v, n, t)])
                if val and val > 0.5:
                    pen = pi[n] * Delta[(n, t)]
                    total_penalty += pen

                    dev = t - tau[n]
                    if dev > 0:
                        flag = f" *** LATE +{dev}d"
                    elif dev < 0:
                        flag = f" *** EARLY {dev}d"
                    else:
                        flag = " ✓ ON TIME"
                    print(f"  Single  : Trip {n} (tau={tau[n]}) → "
                          f"C{r}, V{v}({vtype}), Day {t} | "
                          f"penalty={pen:.1f}€{flag}")
        # combined trips
        for n in trips:
            for k in trips:
                if k == n:
                    continue
                for t in periods:
                    val = pulp.value(y[(r, v, n, k, t)])
                    if val and val > 0.5:
                        pen = pi[n]*Delta[(n,t)] + pi[k]*Delta[(k,t)]
                        total_penalty += pen

                        dev_n = t - tau[n]
                        dev_k = t - tau[k]
                        def fmt(dev):
                            if dev > 0:   return f"LATE +{dev}d"
                            elif dev < 0: return f"EARLY {dev}d"
                            else:         return "ON TIME"
                        flag = f" *** n:{fmt(dev_n)} k:{fmt(dev_k)}"
                        print(f"  Combined: Trip ({n}+{k}) "
                              f"(tau={tau[n]},{tau[k]}) → "
                              f"C{r}, V{v}({vtype}), Day {t} | "
                              f"penalty={pen:.1f}€{flag}")

print(f"\nTotal Time Penalty across all trips: {total_penalty:.2f} €")


Solving Problem 2 (Cooperative + Time-Indexed)...
Status : Optimal
Total Profit: 262.00 €

Carrier 0: S1=92.00  S2=0.00  S3=0.00  Total=92.00  (min S0=92.00)
Carrier 1: S1=48.00  S2=122.00  S3=0.00  Total=170.00  (min S0=132.00)

Trip Assignments:
  Single  : Trip 0 (tau=1) → C0, V2(E), Day 1 | penalty=0.0€ ✓ ON TIME
  Single  : Trip 1 (tau=1) → C0, V2(E), Day 2 | penalty=8.0€ *** LATE +1d
  Single  : Trip 3 (tau=2) → C1, V3(E), Day 2 | penalty=0.0€ ✓ ON TIME
  Combined: Trip (4+2) (tau=3,2) → C1, V4(E), Day 2 | penalty=11.0€ *** n:EARLY -1d k:ON TIME

Total Time Penalty across all trips: 19.00 €
